# DPFE Email Privacy Attack Experiment

Replication of **Huang et al. (2022)** — *Are Large Pre-Trained Language Models Leaking Your Personal Information?*

Fine-tunes **GPT-2 Large** on ENRON emails using LoRA + DP-SGD, then runs a privacy attack to measure email address leakage at different noise levels.

**Runtime:** ~3–4 hours on an A100, ~8–10 hours on a V100/T4.

> **Before running:** Go to `Runtime → Change runtime type` and select **A100** (or at minimum V100) under Hardware accelerator.

> **Resuming after disconnect:** Enable Google Drive in Cell 3 (`USE_DRIVE = True`). The experiment saves a checkpoint after each noise level completes — re-running the notebook will skip already-finished runs automatically.

In [ ]:
# ── 1. Check GPU ─────────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. Go to Runtime → Change runtime type and select a GPU.')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── 2. Clone repo and install dependencies ───────────────────────────────────
import os

REPO_URL = 'https://github.com/usffish/dpfe-email-privacy-experiment.git'
BRANCH   = 'colab'
WORK_DIR = '/content/dpfe-email-privacy-experiment'

if not os.path.exists(WORK_DIR):
    !git clone -b {BRANCH} {REPO_URL} {WORK_DIR}
else:
    print('Repo already cloned, pulling latest...')
    !git -C {WORK_DIR} pull

%cd {WORK_DIR}
!pip install -q -r requirements.txt

In [ ]:
# ── 3. Mount Google Drive for persistence and disconnect recovery ──────────────
# Strongly recommended: saves results and the model cache to Drive so that if
# Colab disconnects, re-running the notebook resumes from the last completed run.
# Set USE_DRIVE = False only if you are certain the session won't disconnect.

USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/dpfe-experiment'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    # Symlink results and model cache into Drive so they survive session resets
    os.makedirs(f'{DRIVE_DIR}/results', exist_ok=True)
    os.makedirs(f'{DRIVE_DIR}/hf_cache', exist_ok=True)
    if not os.path.islink(f'{WORK_DIR}/results'):
        os.symlink(f'{DRIVE_DIR}/results', f'{WORK_DIR}/results')
    os.environ['HF_HOME'] = f'{DRIVE_DIR}/hf_cache'
    print(f'Drive mounted. Results → {DRIVE_DIR}/results')
    print('Checkpoint will be saved after each noise level run.')
else:
    os.makedirs(f'{WORK_DIR}/results', exist_ok=True)
    print('Running without Drive. Results will be lost if the session disconnects.')

In [ ]:
# ── 4. Download ENRON corpus ──────────────────────────────────────────────────
# The real corpus is required — the script will raise an error if it is missing.
# Download takes ~5 minutes.

import os
enron_dir = f'{WORK_DIR}/enron_data'
if not os.path.exists(f'{enron_dir}/maildir'):
    print('Downloading ENRON corpus (~432 MB)...')
    !wget -q https://www.cs.cmu.edu/~enron/enron_mail_20150507.tar.gz -O /tmp/enron.tar.gz
    !mkdir -p {enron_dir}
    !tar -xzf /tmp/enron.tar.gz -C {enron_dir}
    print('Done.')
else:
    print('ENRON corpus already present.')

In [ ]:
# ── 5. Configuration ─────────────────────────────────────────────────────────
# Override any CONFIG values here before running the experiment.
# These are written to a .env file that main.py picks up via python-dotenv.

env_config = {
    'MODEL_NAME':    'gpt2-large',
    'BATCH_SIZE':    '16',
    'EPOCHS':        '3',
    'MAX_EMAILS':    '50000',
    'LEARNING_RATE': '5e-5',
    'SEED':          '42',
}

with open(f'{WORK_DIR}/.env', 'w') as f:
    for k, v in env_config.items():
        f.write(f'{k}={v}\n')

print('Config written to .env:')
for k, v in env_config.items():
    print(f'  {k}={v}')

In [ ]:
# ── 6. Run experiment ─────────────────────────────────────────────────────────
# Streams output live. Expect ~3-4h on A100, ~8-10h on V100/T4.
# If Colab disconnects: re-run all cells from the top — completed noise levels
# are loaded from the Drive checkpoint and skipped automatically.

!python -u main.py

In [ ]:
# ── 7. Display results ────────────────────────────────────────────────────────
import json
import os

results_path = f'{WORK_DIR}/results/table_11_results.json'
if os.path.exists(results_path):
    with open(results_path) as f:
        results = json.load(f)
    print('\nResults saved to:', results_path)
    if USE_DRIVE:
        print('Also persisted to Google Drive.')
else:
    print('Results file not found — experiment may still be running or encountered an error.')